# Preprocesamiento: Paso a Paso
Este notebook recorre el **flujo de trabajo completo de preprocesamiento** para el conjunto de datos de precios de viviendas de King County, explicando cada paso en detalle.

**Nota:** Para uso en producción, consulta el notebook complementario `04b-preprocessing-pipeline.ipynb`, que encapsula todos estos pasos en un pipeline de sklearn reutilizable.

In [ ]:
import numpy as npimport pandas as pdfrom pathlib import Path

## 1. Carga de Datos

In [ ]:
import kagglehubpath = kagglehub.dataset_download("harlfoxem/housesalesprediction")csv_path = Path(path) / "kc_house_data.csv"df = pd.read_csv(csv_path)print(f"Dataset shape: {df.shape}")df.head()

Dataset shape: (21613, 21)


,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


## 2. Parseo de Fechas
La columna `date` contiene fechas de venta como cadenas de texto del tipo `"20140502T000000"`. Necesitamos parsear estas a objetos datetime adecuados.

**⚠️ CRÍTICO:** El parseo de fechas debe ocurrir **ANTES** de dividir los datos. Necesitamos el orden temporal para crear las divisiones de entrenamiento/validación/prueba. El formato de cadena no puede ordenarse cronológicamente.

In [ ]:
# Parse the date cadena to datetime# Format: "YYYYMMDDTHHMMSS" → we only need the first 8 characters (YYYYMMDD)df["date_parsed"] = pd.to_datetime(df["date"].str[:8], format="%Y%m%d")# Sort by date for temporal orderingdf = df.sort_values("date_parsed").reset_index(drop=True)print(f"Date range: {df['date_parsed'].min().date()} to {df['date_parsed'].max().date()}")print(f"Total sales period: {(df['date_parsed'].max() - df['date_parsed'].min()).days} days")

Date range: 2014-05-02 to 2015-05-27
Total sales period: 390 days


## 3. División Temporal Entrenamiento/Validación/Prueba
Usamos **división temporal** (no aleatoria) para prevenir la fuga de datos, como se vio en el notebook anterior.

In [ ]:
def temporal_train_val_test_split(    df: pd.DataFrame,     date_column: str,    val_size: float = 0.15,     test_size: float = 0.15) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:    """    Split data temporally into train, validation, and test sets.        Data is sorted by date_column, then split into three consecutive chunks.        Parameters    ----------    df : DataFrame        Data to split.    date_column : str        Name of the datetime column for temporal ordering.    val_size : float        Proportion for validation set (default 0.15).    test_size : float          Proportion for test set (default 0.15).            Returns    -------    tuple of (train_df, val_df, test_df)    """    df_sorted = df.sort_values(date_column).reset_index(drop=True)        n = len(df_sorted)    train_end = int(n * (1 - val_size - test_size))    val_end = int(n * (1 - test_size))        train_df = df_sorted.iloc[:train_end].copy()    val_df = df_sorted.iloc[train_end:val_end].copy()    test_df = df_sorted.iloc[val_end:].copy()        return train_df, val_df, test_df

In [ ]:
# aplicar the splittrain_df, val_df, test_df = temporal_train_val_test_split(df, date_column="date_parsed")print("Split sizes:")print(f"  Train: {len(train_df):,} records ({len(train_df)/len(df)*100:.1f}%)")print(f"  Val:   {len(val_df):,} records ({len(val_df)/len(df)*100:.1f}%)")print(f"  Test:  {len(test_df):,} records ({len(test_df)/len(df)*100:.1f}%)")print()print("Date ranges:")print(f"  Train: {train_df['date_parsed'].min().date()} to {train_df['date_parsed'].max().date()}")print(f"  Val:   {val_df['date_parsed'].min().date()} to {val_df['date_parsed'].max().date()}")print(f"  Test:  {test_df['date_parsed'].min().date()} to {test_df['date_parsed'].max().date()}")

Split sizes:
  Train: 15,129 records (70.0%)
  Val:   3,242 records (15.0%)
  Test:  3,242 records (15.0%)

Date ranges:
  Train: 2014-05-02 to 2015-01-16
  Val:   2015-01-16 to 2015-03-26
  Test:  2015-03-26 to 2015-05-27


## 4. Ingeniería de Características
Basándonos en los hallazgos del EDA (`01-eda.ipynb`), crearemos varias características derivadas que capturen mejor los patrones subyacentes.

### 4.1 Característica Temporal: `days_since_start`
Esta característica captura las **tendencias del mercado a lo largo del tiempo** — los precios inmobiliarios tienden a apreciarse (o depreciarse) sistemáticamente.

Calculamos: `days_since_start = fecha_de_venta - fecha_de_referencia`

In [ ]:
train_min_date = train_df["date_parsed"].min()print(f"Reference date (from training): {train_min_date.date()}")

Reference date (from training): 2014-05-02


Este valor necesita almacenarse, ya que luego se usará para datos nuevos en un pipeline de producción para calcular la característica de ingeniería `days_since_start`. Será un **parámetro ajustado**.

In [ ]:
# aplicar the SAME reference date to ALL splitstrain_df["days_since_start"] = (train_df["date_parsed"] - train_min_date).dt.daysval_df["days_since_start"] = (val_df["date_parsed"] - train_min_date).dt.daystest_df["days_since_start"] = (test_df["date_parsed"] - train_min_date).dt.daysprint("days_since_start ranges (should increase across splits):")print(f"  Train: {train_df['days_since_start'].min():3d} to {train_df['days_since_start'].max():3d}")print(f"  Val:   {val_df['days_since_start'].min():3d} to {val_df['days_since_start'].max():3d}")print(f"  Test:  {test_df['days_since_start'].min():3d} to {test_df['days_since_start'].max():3d}")

days_since_start ranges (should increase across splits):
  Train:   0 to 259
  Val:   259 to 328
  Test:  328 to 390


**Verificar:** Los rangos de validación y prueba deben ser **mayores** que los de entrenamiento (están en el futuro). Si ves valores que comienzan desde 0 en cada división, ¡tienes fuga de datos!

### 4.2 Características de Edad
Estas características capturan las propiedades relacionadas con la antigüedad de la propiedad.

In [ ]:
def add_age_features(df: pd.DataFrame) -> pd.DataFrame:    """Add age-related features (per-row computation, no fitting needed)."""    df = df.copy()        sale_year = df["date_parsed"].dt.year        # House age at time of sale    df["house_age"] = sale_year - df["yr_built"]        # Was the house ever renovated?    df["was_renovated"] = (df["yr_renovated"] > 0).astype(int)        # Years since last renovation (0 if never renovated)    df["years_since_renovation"] = np.where(        df["yr_renovated"] > 0,        sale_year - df["yr_renovated"],        0    )        return df# aplicar to all splitstrain_df = add_age_features(train_df)val_df = add_age_features(val_df)test_df = add_age_features(test_df)print("Age features added: house_age, was_renovated, years_since_renovation")

Age features added: house_age, was_renovated, years_since_renovation


### 4.3 Características de Razón
Estas características comparan propiedades consigo mismas o con sus vecinos.

In [ ]:
def add_ratio_features(df: pd.DataFrame) -> pd.DataFrame:    """Add ratio features (per-row computation, no fitting needed)."""    df = df.copy()        # What fraction of living space is basement?    df["basement_ratio"] = df["sqft_basement"] / df["sqft_living"].replace(0, 1)        # How does living area compare to neighbors?    df["living_vs_neighbors"] = df["sqft_living"] / df["sqft_living15"].replace(0, 1)        # How does lot size compare to neighbors?    df["lot_vs_neighbors"] = df["sqft_lot"] / df["sqft_lot15"].replace(0, 1)        return df# aplicar to all splitstrain_df = add_ratio_features(train_df)val_df = add_ratio_features(val_df)test_df = add_ratio_features(test_df)print("Ratio features added: basement_ratio, living_vs_neighbors, lot_vs_neighbors")

Ratio features added: basement_ratio, living_vs_neighbors, lot_vs_neighbors


## 5. Eliminación de Columnas No Predictivas
Algunas columnas deben eliminarse porque:
- Son identificadores (no predictivos)
- Han sido reemplazadas por características derivadas
- No pueden usarse de forma efectiva (alta cardinalidad)

In [ ]:
columns_to_drop = [    "id",           # Property identifier (21,000+ unique valores, not predictive)    "date",         # Original cadena format (replaced by date_parsed)    "date_parsed",  # Used for splitting, now replaced by days_since_start    "zipcode",      # High cardinality categorical (we use lat/long instead)    "yr_built",     # Replaced by house_age    "yr_renovated", # Replaced by was_renovated, years_since_renovation]train_df = train_df.drop(columns=columns_to_drop)val_df = val_df.drop(columns=columns_to_drop)test_df = test_df.drop(columns=columns_to_drop)print(f"Columns after dropping: {list(train_df.columns)}")print(f"Shape: {train_df.shape}")

Columns after dropping: ['price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'lat', 'long', 'sqft_living15', 'sqft_lot15', 'days_since_start', 'house_age', 'was_renovated', 'years_since_renovation', 'basement_ratio', 'living_vs_neighbors', 'lot_vs_neighbors']
Shape: (15129, 23)


## 6. Separación de Características y Objetivo

In [ ]:
target = "price"X_train = train_df.drop(columns=[target])y_train = train_df[target]X_val = val_df.drop(columns=[target])y_val = val_df[target]X_test = test_df.drop(columns=[target])y_test = test_df[target]print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")print(f"X_val:   {X_val.shape}, y_val:   {y_val.shape}")print(f"X_test:  {X_test.shape}, y_test:  {y_test.shape}")

X_train: (15129, 22), y_train: (15129,)
X_val:   (3242, 22), y_val:   (3242,)
X_test:  (3242, 22), y_test:  (3242,)


## 7. Preprocesamiento Numérico
La mayoría de los algoritmos de Machine Learning funcionan mejor con características normalizadas/estandarizadas. Aplicaremos:
1. **Transformación logarítmica** para características muy sesgadas (superficie en pies cuadrados)
2. **Escalado estándar** para todas las características numéricas

In [ ]:
from sklearn.preprocessing import StandardScalerfrom sklearn.compose import ColumnTransformerfrom sklearn.pipeline import Pipelinefrom sklearn.preprocessing import FunctionTransformer

In [ ]:
# Identify característica groupslog_features = [    "sqft_living", "sqft_lot", "sqft_above", "sqft_basement",     "sqft_living15", "sqft_lot15"]# Binary características that don't need scalingpassthrough_features = ["waterfront", "was_renovated"]# All other características get standard scalingscale_features = [col for col in X_train.columns                   if col not in log_features + passthrough_features]print(f"Log+scale features ({len(log_features)}): {log_features}")print(f"Scale only ({len(scale_features)}): {scale_features}")print(f"Passthrough ({len(passthrough_features)}): {passthrough_features}")

Log+scale features (6): ['sqft_living', 'sqft_lot', 'sqft_above', 'sqft_basement', 'sqft_living15', 'sqft_lot15']
Scale only (14): ['bedrooms', 'bathrooms', 'floors', 'view', 'condition', 'grade', 'lat', 'long', 'days_since_start', 'house_age', 'years_since_renovation', 'basement_ratio', 'living_vs_neighbors', 'lot_vs_neighbors']
Passthrough (2): ['waterfront', 'was_renovated']


In [ ]:
# Create preprocessing pipelinelog_pipeline = Pipeline([    ("log", FunctionTransformer(np.log1p, validate=True, feature_names_out="one-to-one")),    ("scale", StandardScaler())])preprocessor = ColumnTransformer([    ("log", log_pipeline, log_features),    ("scale", StandardScaler(), scale_features),    ("passthrough", "passthrough", passthrough_features)])# FIT ON entrenamiento ONLYpreprocessor.fit(X_train)print("Preprocessor fitted on training data.")

Preprocessor fitted on training data.


In [ ]:
# Transform all splits using the fitted preprocessorX_train_processed = preprocessor.transform(X_train)X_val_processed = preprocessor.transform(X_val)X_test_processed = preprocessor.transform(X_test)print(f"Processed shapes:")print(f"  Train: {X_train_processed.shape}")print(f"  Val:   {X_val_processed.shape}")print(f"  Test:  {X_test_processed.shape}")

Processed shapes:
  Train: (15129, 22)
  Val:   (3242, 22)
  Test:  (3242, 22)


In [ ]:
# Get característica names from preprocessorfeature_names = preprocessor.get_feature_names_out()print(f"Total features: {len(feature_names)}")print(f"Feature names: {list(feature_names)}")

Total features: 22
Feature names: ['log__sqft_living', 'log__sqft_lot', 'log__sqft_above', 'log__sqft_basement', 'log__sqft_living15', 'log__sqft_lot15', 'scale__bedrooms', 'scale__bathrooms', 'scale__floors', 'scale__view', 'scale__condition', 'scale__grade', 'scale__lat', 'scale__long', 'scale__days_since_start', 'scale__house_age', 'scale__years_since_renovation', 'scale__basement_ratio', 'scale__living_vs_neighbors', 'scale__lot_vs_neighbors', 'passthrough__waterfront', 'passthrough__was_renovated']


## 8. Verificación del Preprocesamiento
Hagamos una verificación de cordura de nuestro preprocesamiento para detectar cualquier problema.

In [ ]:
# Check that scaling worked (entrenamiento should have media~0, std~1)train_means = X_train_processed.mean(axis=0)train_stds = X_train_processed.std(axis=0)print("Training set statistics (should be ~0 mean, ~1 std for scaled features):")print(f"  Means range: {train_means.min():.3f} to {train_means.max():.3f}")print(f"  Stds range:  {train_stds.min():.3f} to {train_stds.max():.3f}")

Training set statistics (should be ~0 mean, ~1 std for scaled features):
  Means range: -0.000 to 0.045
  Stds range:  0.088 to 1.000


In [ ]:
# Check for any NaN or Inf valoresfor name, data in [("Train", X_train_processed), ("Val", X_val_processed), ("Test", X_test_processed)]:    nan_count = np.isnan(data).sum()    inf_count = np.isinf(data).sum()    print(f"{name}: {nan_count} NaN, {inf_count} Inf")

Train: 0 NaN, 0 Inf
Val: 0 NaN, 0 Inf
Test: 0 NaN, 0 Inf

✓ Preprocessing verification complete


## 9. Guardar Datos Preprocesados
Guardamos:
1. Arrays de características procesadas y objetivos
2. El preprocesador ajustado (para escalar nuevos datos de manera consistente)
3. Valores de referencia necesarios para la ingeniería de características
4. Nombres de características para interpretabilidad

In [ ]:
import jobliboutput_dir = Path("processed_data")output_dir.mkdir(exist_ok=True)# Save processed arraysnp.save(output_dir / "X_train.npy", X_train_processed)np.save(output_dir / "X_val.npy", X_val_processed)np.save(output_dir / "X_test.npy", X_test_processed)np.save(output_dir / "y_train.npy", y_train.values)np.save(output_dir / "y_val.npy", y_val.values)np.save(output_dir / "y_test.npy", y_test.values)# Save the fitted preprocessor (for scaling)joblib.dump(preprocessor, output_dir / "scaler.joblib")# Save ingeniería de características reference valoresreference_values = {    "train_min_date": train_min_date,}joblib.dump(reference_values, output_dir / "reference_values.joblib")# Save característica namesnp.save(output_dir / "feature_names.npy", feature_names)print(f"Saved preprocessed data to {output_dir.absolute()}")

Saved preprocessed data to /media/DIURNOext4/alejandro/wip-clase/PIA-SAA/examen-SAA/example_repos/king-county/processed_data
